In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "albiach2015comparing")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "22 05 04_Albiach-Serrano et al., 2015.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)

In [3]:
df['study_id']="albiach2015comparing"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"subject": "ape",
    "sex":"sex_original",
    "continuousstrip":"continuous_strip"}, inplace=True)

In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")


df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')


In [5]:
df.dropna(subset=['species'], inplace=True)
# df.columns
df.rename(columns={"ape": "participant",
                'age_years':'age_in_years',
                'age_months':'age_original'}, inplace=True)

space_list = ['rearing','order']
for x in space_list:
    df[x].replace(' ', '_', inplace=True, regex=True)

In [6]:
albiach2015comparing_standardized=df[[ 'study_id','participant', 'age_original', 'age_in_years', 'sex', 'species',
         'session', 'trial', 'condition', 'order', 'continuous_strip',
       'choice', 'correct' ]]
comp_out_path_stand = os.path.join(out_pathway, 'albiach2015comparing_standardized.csv')
albiach2015comparing_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =albiach2015comparing_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
albiach2015comparing_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'albiach2015comparing_glossary.csv')
albiach2015comparing_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
